# EEG-PRO Zenodo Download & ICA Preprocessing Pipeline (v2)
**Tinnitus EEG Phenotyping — IEEE Paper Support**

### Purpose
This notebook addresses the ICA artifact-rejection gap identified in the IEEE paper review:
- Boundary sensors E1, E127, E32, E128, E125 are in EOG/EMG-susceptible regions
- The `n=20` minor phenotype must be confirmed as neurological, not artifact-driven

### Pipeline Steps
1. Install dependencies
2. Mount Google Drive (persistent storage)
3. Download Zenodo Record 13219018
4. Run memory-efficient ICA loop with ICLabel auto-labelling
5. Inspect artifact log
6. Bridge to Riemannian covariance pipeline

In [ ]:
# Cell 1: Install dependencies
!pip install -q mne mne-icalabel pyriemann zenodo_get
print('All dependencies installed.')

In [ ]:
# Cell 2: Mount Google Drive (saves data across session restarts)
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_BASE = '/content/drive/MyDrive/Tinnitus_ICA'
RAW_DIR    = os.path.join(DRIVE_BASE, 'raw_zenodo')
CLEAN_DIR  = os.path.join(DRIVE_BASE, 'clean_eeg_data')

os.makedirs(RAW_DIR,   exist_ok=True)
os.makedirs(CLEAN_DIR, exist_ok=True)

print(f'Raw data  -> {RAW_DIR}')
print(f'Clean data -> {CLEAN_DIR}')

In [ ]:
# Cell 3: Download Zenodo EEG-PRO (Record 13219018)
os.chdir(RAW_DIR)
!zenodo_get 13219018

files = os.listdir(RAW_DIR)
print(f'\nDownloaded {len(files)} items:')
for f in sorted(files)[:20]:
    print(' ', f)

In [ ]:
# Cell 3b: Unzip archives if present (run only if you see .zip files above)
import glob, zipfile

for zf in glob.glob(os.path.join(RAW_DIR, '*.zip')):
    print(f'Extracting {zf} ...')
    with zipfile.ZipFile(zf, 'r') as z:
        z.extractall(RAW_DIR)

set_files = glob.glob(os.path.join(RAW_DIR, '**', '*.set'), recursive=True)
print(f'\nFound {len(set_files)} .set files')

In [ ]:
# Cell 4: Production ICA Loop
import mne
import glob
import os
import csv
from mne.preprocessing import ICA
from mne_icalabel import label_components

mne.set_log_level('WARNING')

set_files = sorted(glob.glob(os.path.join(RAW_DIR, '**', '*.set'), recursive=True))
print(f'Found {len(set_files)} .set files to process.')

log_path   = os.path.join(CLEAN_DIR, 'ica_artifact_log.csv')
log_exists = os.path.exists(log_path)

with open(log_path, 'a', newline='') as logfile:
    writer = csv.writer(logfile)
    if not log_exists:
        writer.writerow(['subject_file', 'n_channels', 'n_components_fit',
                         'n_eye_removed', 'n_muscle_removed', 'status'])

    for file in set_files:
        subj_name = os.path.basename(file)
        out_name  = os.path.join(CLEAN_DIR, subj_name.replace('.set', '-ica-raw.fif'))

        if os.path.exists(out_name):
            print(f'  [SKIP] {subj_name} — already processed.')
            continue

        print(f'\n--- Processing: {subj_name} ---')
        try:
            # A. Load
            raw = mne.io.read_raw_eeglab(file, preload=True, verbose=False)
            n_ch = len(raw.ch_names)
            print(f'  Channels: {n_ch}')

            # B. Skip non-EGI 128-ch recordings
            if n_ch < 100:
                print(f'  [SKIP] Only {n_ch} channels — not 128-ch EGI net.')
                writer.writerow([subj_name, n_ch, 0, 0, 0, 'SKIPPED_CH_COUNT'])
                del raw
                continue

            # C. Set EGI montage for ICLabel spatial features
            try:
                montage = mne.channels.make_standard_montage('GSN-HydroCel-128')
                raw.set_montage(montage, on_missing='ignore', verbose=False)
            except Exception:
                pass

            # D. Filter: 1 Hz HP (ICA requirement) + 45 Hz LP (matches paper)
            raw.filter(l_freq=1.0, h_freq=45.0, verbose=False)

            # E. Fit ICA (99% variance, FastICA)
            ica = ICA(n_components=0.99, method='fastica',
                      max_iter=500, random_state=42)
            ica.fit(raw)
            n_fit = ica.n_components_
            print(f'  ICA fitted: {n_fit} components')

            # F. ICLabel auto-classification
            ic_labels = label_components(raw, ica, method='iclabel')
            labels = ic_labels['labels']
            probs  = ic_labels['y_pred_proba']

            # G. Conservative rejection threshold: 80% confidence
            THRESHOLD  = 0.80
            eye_idx    = [i for i, (lbl, p) in enumerate(zip(labels, probs))
                          if lbl == 'eye'    and p.max() > THRESHOLD]
            muscle_idx = [i for i, (lbl, p) in enumerate(zip(labels, probs))
                          if lbl == 'muscle' and p.max() > THRESHOLD]
            exclude_idx = eye_idx + muscle_idx

            print(f'  Removing: {len(eye_idx)} eye + {len(muscle_idx)} muscle components')
            ica.exclude = exclude_idx

            # H. Reconstruct clean signal
            raw_clean = ica.apply(raw.copy())

            # I. Save to Drive
            raw_clean.save(out_name, overwrite=True, verbose=False)
            print(f'  [SAVED] -> {out_name}')

            writer.writerow([subj_name, n_ch, n_fit,
                             len(eye_idx), len(muscle_idx), 'OK'])

            # J. Free RAM
            del raw, ica, raw_clean, ic_labels

        except Exception as e:
            print(f'  [ERROR] {subj_name}: {e}')
            writer.writerow([subj_name, -1, -1, -1, -1, f'ERROR: {e}'])
            continue

print('\n=== ICA LOOP COMPLETE ===')
print(f'Cleaned files -> {CLEAN_DIR}')
print(f'Artifact log  -> {log_path}')

In [ ]:
# Cell 5: Review artifact log
import pandas as pd

log_df = pd.read_csv(log_path)
print('Status breakdown:')
print(log_df['status'].value_counts())
print(f'\nSuccessful: {(log_df["status"]=="OK").sum()} / {len(log_df)}')

ok = log_df[log_df['status'] == 'OK']
print(f'\nAvg eye components removed:    {ok["n_eye_removed"].mean():.2f}')
print(f'Avg muscle components removed: {ok["n_muscle_removed"].mean():.2f}')

high = ok[(ok['n_eye_removed'] + ok['n_muscle_removed']) > 10]
print(f'\nSubjects with >10 artifact components: {len(high)}')
print(high[['subject_file', 'n_eye_removed', 'n_muscle_removed']])

In [ ]:
# Cell 6: Bridge to Riemannian Pipeline
# Load ICA-cleaned .fif files and compute OAS covariance matrices
import mne, glob, numpy as np
from pyriemann.estimation import Covariances

mne.set_log_level('WARNING')

clean_files = sorted(glob.glob(os.path.join(CLEAN_DIR, '*-ica-raw.fif')))
print(f'Found {len(clean_files)} ICA-cleaned subjects.')

BANDS = {
    'broadband': (1,  45),
    'delta':     (1,   4),
    'theta':     (4,   8),
    'alpha':     (8,  13),
    'beta':      (13, 30),
    'gamma':     (30, 45),
}

all_subjects_cov = []
valid_subjects   = []

for fif_file in clean_files:
    try:
        raw = mne.io.read_raw_fif(fif_file, preload=True, verbose=False)
        if len(raw.ch_names) < 100:
            continue

        subject_covs = {}
        for band_name, (l, h) in BANDS.items():
            rb = raw.copy().filter(l, h, verbose=False)
            epochs = mne.make_fixed_length_epochs(rb, duration=2.0,
                                                   preload=True, verbose=False)
            if len(epochs) == 0:
                break
            covs = Covariances(estimator='oas').fit_transform(epochs.get_data())
            subject_covs[band_name] = covs.mean(axis=0)

        if len(subject_covs) == len(BANDS):
            all_subjects_cov.append(subject_covs)
            valid_subjects.append(fif_file)
        del raw

    except Exception as e:
        print(f'Error: {fif_file}: {e}')

print(f'\nValid subjects for Riemannian pipeline: {len(valid_subjects)}')
print('Ready to feed into TangentSpace + k-means pipeline.')